# Notebook 3 — Train H-HFGAT Model

**Input**: outputs từ Notebook 1 (`embeddings/`) và Notebook 2 (`matrices/`, `subsample/`)

**Output**: `best_model.pt`, training curves, test metrics

Pipeline:
1. Load embeddings & matrices
2. Tạo FITB files & split train/val/test cho recommendation
3. Định nghĩa model H_HFGAT
4. Train với joint loss (BPR rec + BPR compat)
5. Evaluate trên test set

## 0. Cài đặt

In [1]:
import subprocess, sys

def pip_install(pkg, extra_args=None):
    cmd = [sys.executable, '-m', 'pip', 'install', pkg, '-q']
    if extra_args:
        cmd.extend(extra_args)
    subprocess.check_call(cmd)

pip_install('torch_geometric')
import torch
cuda_version = torch.version.cuda
torch_version = torch.__version__.split('+')[0]
print(f'PyTorch: {torch.__version__}')
if cuda_version:
    pip_install('torch-scatter', ['-f', f'https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version.replace(".", "")}.html'])
else:
    try:
        pip_install('torch-scatter')
    except subprocess.CalledProcessError:
        print('⚠️ torch-scatter skipped; PyTorch fallback will be used')
print('✅ Install OK')


PyTorch: 2.12.0
✅ Install OK


## 1. Imports & Setup

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import os, time, random, json, copy
from collections import defaultdict
from sklearn.metrics import roc_auc_score
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from scipy.sparse import load_npz

try:
    from torch_scatter import scatter_add
except Exception:
    def scatter_add(src, index, dim=0, dim_size=None, out=None):
        if dim_size is None:
            dim_size = int(index.max().item()) + 1
        shape = list(src.shape)
        shape[dim] = dim_size
        result = src.new_zeros(shape)
        idx = index
        for _ in range(src.dim() - 1):
            idx = idx.unsqueeze(-1)
        idx = idx.expand_as(src)
        result.scatter_add_(dim, idx, src)
        return result

import sys
from pathlib import Path
REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import fgat_config as cfg

SEED = cfg.RANDOM_SEED
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
PIN_MEMORY = device.type == 'cuda'  # MPS/CPU: pin_memory not supported
print(f'Device: {device}  pin_memory={PIN_MEMORY}')


Device: mps  pin_memory=False


/opt/homebrew/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib
import fgat_config as cfg
importlib.reload(cfg)  # pick up edits to fgat_config.py without kernel restart
from fgat_config import resolve_paths, print_config_summary

_paths = resolve_paths(REPO_ROOT)
OUTPUT_PATH = _paths['OUTPUT_PATH']
BEST_STATE_PATH = _paths['BEST_STATE_PATH']

# Training hyperparameters (from fgat_config.py)
EPOCHS = cfg.EPOCHS
LR = cfg.LR
WEIGHT_DECAY = cfg.WEIGHT_DECAY
LAMBDA_COMP = cfg.LAMBDA_COMP
BATCH_SIZE = cfg.BATCH_SIZE
NEG_PER_POS = cfg.NEG_PER_POS
PATIENCE = cfg.PATIENCE
EVAL_EVERY = cfg.EVAL_EVERY
EARLY_STOP_METRIC = cfg.EARLY_STOP_METRIC
TOP_K = cfg.TOP_K
EVAL_NEG_SAMPLES = cfg.EVAL_NEG_SAMPLES
EMBED_DIM = cfg.EMBED_DIM
NUM_HEADS = cfg.NUM_HEADS
DROPOUT = cfg.DROPOUT
MAX_OUTFIT_ITEMS_FOR_COMP = cfg.MAX_OUTFIT_ITEMS_FOR_COMP
SPLIT_MODE = cfg.SPLIT_MODE
SCHEDULER_PATIENCE = cfg.SCHEDULER_PATIENCE
LEARNABLE_EMBEDDINGS = cfg.LEARNABLE_EMBEDDINGS
COMPAT_BPR_MARGIN = cfg.COMPAT_BPR_MARGIN
COMPAT_LR_MULT = cfg.COMPAT_LR_MULT
COMPAT_DETACH_INPUT = cfg.COMPAT_DETACH_INPUT

NB1_CANDIDATES = [
    str(REPO_ROOT / 'output_fgat_active_user') + '/',
    '/kaggle/input/notebooks/kiettruonglifeez/fgat-session1-active-user/',
    '/kaggle/working/',
]
NB2_CANDIDATES = [
    str(REPO_ROOT / 'output_fgat_active_user') + '/',
    '/kaggle/input/notebooks/kiettruonglifeez/fgat-session2-active-user/',
    '/kaggle/working/',
]

NB1_DIR = next((p for p in NB1_CANDIDATES if os.path.exists(p + 'embeddings/item_embeddings.npy')), None)
NB2_DIR = next((p for p in NB2_CANDIDATES if os.path.exists(p + 'matrices/item_item_matrix.npz')), None)
assert NB1_DIR, '❌ Chạy Notebook 1 trước!'
assert NB2_DIR, '❌ Chạy Notebook 2 trước!'

EMB_DIR = NB1_DIR + 'embeddings/'
SUB_DIR = NB1_DIR + 'subsample/'
MTX_DIR = NB2_DIR + 'matrices/'
os.makedirs(OUTPUT_PATH + 'splits/', exist_ok=True)
os.makedirs(OUTPUT_PATH + 'models/', exist_ok=True)

print_config_summary()
print(f'  NB1_DIR: {NB1_DIR}')
print(f'  NB2_DIR: {NB2_DIR}')
print('✅ Paths OK')


── fgat_config ──
  MIN_USER_INTERACTIONS=4  SPLIT_MODE=per_user
  MIN_TOP_NEIGHBORS=10  FORCE_REBUILD_ITEM_ITEM_MATRIX=True
  EPOCHS=50  LR=0.001  WEIGHT_DECAY=1e-05  LAMBDA_COMP=0.3
  COMPAT_DETACH_INPUT=True  COMPAT_BPR_MARGIN=0.0  COMPAT_LR_MULT=1.0
  BATCH_SIZE=512  NEG_PER_POS=3  PATIENCE=10
  SCHEDULER_PATIENCE=5  LEARNABLE_EMBEDDINGS=True
  EVAL_EVERY=1  EARLY_STOP_METRIC=HR@K
  IMAGE_BATCH_SIZE=64  TEXT_BATCH_SIZE=128
  MAX_TEXT_LENGTH=ad-hoc (BERT max)  FORCE_REBUILD_FEATURES=True
  NB1_DIR: /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/output_fgat_active_user/
  NB2_DIR: /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/output_fgat_active_user/
✅ Paths OK


## Stage 4A — Load subsample metadata

In [4]:
print('=== STAGE 4A: Load subsample metadata ===')

item_data   = pd.read_csv(SUB_DIR + 'item_sub.csv')
outfit_data = pd.read_csv(SUB_DIR + 'outfit_sub.csv')
user_data   = pd.read_csv(SUB_DIR + 'user_sub.csv')
train_uo    = pd.read_csv(SUB_DIR + 'train_uo_sub.csv')

item_data['item_id']     = item_data['item_id'].astype(int)
outfit_data['outfit_id'] = outfit_data['outfit_id'].astype(int)
user_data['user_id']     = user_data['user_id'].astype(int)
train_uo['user_id']      = train_uo['user_id'].astype(int)
train_uo['outfit_id']    = train_uo['outfit_id'].astype(int)

if os.path.exists(SUB_DIR + 'filter_stats.json'):
    with open(SUB_DIR + 'filter_stats.json') as f:
        stats = json.load(f)
    print(f'  Filter stats: {stats}')

# ID → index maps (0-based, dùng xuyên suốt)
item_ids_sorted   = sorted(item_data['item_id'].unique())
outfit_ids_sorted = sorted(outfit_data['outfit_id'].unique())
user_ids_sorted   = sorted(user_data['user_id'].unique())

item2id   = {iid: idx for idx, iid in enumerate(item_ids_sorted)}
outfit2id = {oid: idx for idx, oid in enumerate(outfit_ids_sorted)}
user2id   = {uid: idx for idx, uid in enumerate(user_ids_sorted)}

N_ITEMS   = len(item2id)
N_OUTFITS = len(outfit2id)
N_USERS   = len(user2id)

print(f'  Items  : {N_ITEMS:,}')
print(f'  Outfits: {N_OUTFITS:,}')
print(f'  Users  : {N_USERS:,}')
print(f'  Edges  : {len(train_uo):,}')
print('✅ Stage 4A hoàn thành!')

=== STAGE 4A: Load subsample metadata ===
  Filter stats: {'min_user_interactions': 4, 'items': 14419, 'outfits': 6622, 'users': 127536, 'edges': 127536}
  Items  : 14,419
  Outfits: 6,622
  Users  : 25,263
  Edges  : 127,536
✅ Stage 4A hoàn thành!


## Stage 4B — Load embeddings & graph matrices

In [5]:
print('=== STAGE 4B: Load embeddings & graph matrices ===')

def load_npz_edge_matrix(file_path):
    data = sp.load_npz(file_path)
    row, col = data.nonzero()
    edge_index = torch.tensor(np.vstack((row, col)), dtype=torch.long).to(device)
    edge_weight = torch.tensor(
        np.array(data[row, col]).flatten(), dtype=torch.float32
    ).to(device) if data.nnz > 0 else None
    return edge_index, edge_weight

# ── Embeddings (cột 0 = ID, cột 1: = embedding dims) ─────────────────
item_embs   = F.normalize(
    torch.tensor(np.load(EMB_DIR + 'item_embeddings.npy')[:, 1:], dtype=torch.float32), dim=-1
).to(device)
outfit_embs = F.normalize(
    torch.tensor(np.load(EMB_DIR + 'outfit_embeddings.npy')[:, 1:], dtype=torch.float32), dim=-1
).to(device)
user_embs   = F.normalize(
    torch.tensor(np.load(EMB_DIR + 'user_embeddings.npy')[:, 1:], dtype=torch.float32), dim=-1
).to(device)

print(f'  item_embs  : {item_embs.shape}')
print(f'  outfit_embs: {outfit_embs.shape}')
print(f'  user_embs  : {user_embs.shape}')

# ── Graph edge matrices ───────────────────────────────────────────────
item_item_index,   item_item_weight   = load_npz_edge_matrix(MTX_DIR + 'item_item_matrix.npz')
outfit_item_index, outfit_item_weight = load_npz_edge_matrix(MTX_DIR + 'outfit_item_adj.npz')

# NOTE: user_outfit_index (full) chỉ dùng để train - sẽ bị thay bằng train-only ở Stage 4C
# Tạm load để dimension check, sẽ override sau khi split xong
# NOTE: user_outfit_adj.npz (full) chỉ load tạm để check dimension.
# Train-only graph (user_outfit_train_index) sẽ được build ở Stage 4C sau khi split.
user_outfit_index, user_outfit_weight = load_npz_edge_matrix(MTX_DIR + 'user_outfit_adj.npz')

print(f'  item_item_index  : {item_item_index.shape}  nnz={item_item_index.shape[1]:,}')
print(f'  outfit_item_index: {outfit_item_index.shape} nnz={outfit_item_index.shape[1]:,}')
print(f'  user_outfit_index (full): {user_outfit_index.shape} nnz={user_outfit_index.shape[1]:,}')

# ── Dimension check ───────────────────────────────────────────────────
assert item_embs.shape[0] == N_ITEMS,   f'item_embs rows {item_embs.shape[0]} != N_ITEMS {N_ITEMS}'
assert outfit_embs.shape[0] == N_OUTFITS, f'outfit_embs rows {outfit_embs.shape[0]} != N_OUTFITS {N_OUTFITS}'
assert user_embs.shape[0] == N_USERS,   f'user_embs rows {user_embs.shape[0]} != N_USERS {N_USERS}'
print('  Dimension check: ✅')
print('✅ Stage 4B hoàn thành!')


=== STAGE 4B: Load embeddings & graph matrices ===
  item_embs  : torch.Size([14419, 64])
  outfit_embs: torch.Size([6622, 64])
  user_embs  : torch.Size([25263, 64])
  item_item_index  : torch.Size([2, 70338])  nnz=70,338
  outfit_item_index: torch.Size([2, 26047]) nnz=26,047
  user_outfit_index (full): torch.Size([2, 127536]) nnz=127,536
  Dimension check: ✅
✅ Stage 4B hoàn thành!


## Stage 4C — Tạo split files

In [6]:
print('=== STAGE 4C: Tạo split files ===')

TRAIN_REC_FILE = OUTPUT_PATH + 'splits/train_rec.txt'
VAL_REC_FILE   = OUTPUT_PATH + 'splits/val_rec.txt'
TEST_REC_FILE  = OUTPUT_PATH + 'splits/test_rec.txt'
TRAIN_FITB     = OUTPUT_PATH + 'splits/train_fitb.txt'
VAL_FITB       = OUTPUT_PATH + 'splits/val_fitb.txt'
TEST_FITB      = OUTPUT_PATH + 'splits/test_fitb.txt'

# ── Xóa file cũ nếu có để tạo lại từ đầu ────────────────────────────
for f in [TRAIN_REC_FILE, VAL_REC_FILE, TEST_REC_FILE,
          TRAIN_FITB, VAL_FITB, TEST_FITB]:
    if os.path.exists(f):
        os.remove(f)
        print(f'  🗑️  Đã xóa file cũ: {os.path.basename(f)}')

# ── Split recommendation: per-user stratified 80/10/10 ───────────────
if not os.path.exists(TRAIN_REC_FILE):
    user_groups = train_uo.groupby('user_id')['outfit_id'].apply(list).reset_index()
    train_lines, val_lines, test_lines = [], [], []
    n_only_train = 0

    # Lưu mapping user→train_outfits để build train-only graph
    user_train_outfits = {}

    for _, row in user_groups.iterrows():
        uid  = int(row['user_id'])
        oids = list(row['outfit_id'])
        random.shuffle(oids)
        n = len(oids)

        if n < 3:
            train_lines.append(f"{uid} {' '.join(map(str, oids))}\n")
            user_train_outfits[uid] = oids
            n_only_train += 1
            continue

        n_test = max(1, round(n * 0.1))
        n_val  = max(1, round(n * 0.1))
        if n_val + n_test >= n:
            n_val  = 1
            n_test = 1

        tr = oids[:n - n_val - n_test]
        va = oids[n - n_val - n_test: n - n_test]
        te = oids[n - n_test:]

        if tr:
            train_lines.append(f"{uid} {' '.join(map(str, tr))}\n")
            user_train_outfits[uid] = tr
        if va: val_lines.append(f"{uid} {' '.join(map(str, va))}\n")
        if te: test_lines.append(f"{uid} {' '.join(map(str, te))}\n")

    with open(TRAIN_REC_FILE, 'w') as f: f.writelines(train_lines)
    with open(VAL_REC_FILE,   'w') as f: f.writelines(val_lines)
    with open(TEST_REC_FILE,  'w') as f: f.writelines(test_lines)

    # ── Đếm interactions thay vì users ───────────────────────────────
    def count_interactions(lines):
        return sum(len(line.strip().split()) - 1 for line in lines)

    n_tr_int = count_interactions(train_lines)
    n_va_int = count_interactions(val_lines)
    n_te_int = count_interactions(test_lines)
    total    = n_tr_int + n_va_int + n_te_int

    print(f'  Rec split (users)        → train: {len(train_lines):,} | val: {len(val_lines):,} | test: {len(test_lines):,}')
    print(f'  Rec split (interactions) → train: {n_tr_int:,} ({n_tr_int/total*100:.1f}%) | '
          f'val: {n_va_int:,} ({n_va_int/total*100:.1f}%) | '
          f'test: {n_te_int:,} ({n_te_int/total*100:.1f}%)')
    print(f'  (trong đó {n_only_train:,} users chỉ có <3 interactions → toàn bộ vào train)')
else:
    print('  Rec split files đã tồn tại, đọc lại train split...')
    user_train_outfits = {}
    with open(TRAIN_REC_FILE) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2: continue
            uid = int(parts[0])
            user_train_outfits[uid] = [int(x) for x in parts[1:]]

# ── FIX: Build train-only user_outfit graph (loại bỏ val/test edges) ──
print('\n  🔧 Building train-only user_outfit graph (no val/test edges)...')
from scipy.sparse import dok_matrix, save_npz as sp_save_npz

train_uo_adj = dok_matrix((N_USERS, N_OUTFITS), dtype=np.float32)

# Rebuild sparse matrix chỉ từ train split
for uid, oids in user_train_outfits.items():
    u_idx = user2id.get(uid, -1)
    if u_idx == -1: continue
    for oid in oids:
        o_idx = outfit2id.get(oid, -1)
        if o_idx != -1:
            train_uo_adj[u_idx, o_idx] = 1.0

train_uo_adj_csr = train_uo_adj.tocsr()
TRAIN_UO_ADJ_FILE = OUTPUT_PATH + 'splits/user_outfit_adj_train.npz'
sp_save_npz(TRAIN_UO_ADJ_FILE, train_uo_adj_csr)

# Override user_outfit_index bằng train-only graph
row_t, col_t = train_uo_adj_csr.nonzero()
user_outfit_train_index  = torch.tensor(np.vstack((row_t, col_t)), dtype=torch.long).to(device)
user_outfit_train_weight = torch.ones(len(row_t), dtype=torch.float32).to(device)

print(f'  user_outfit_index (full) : nnz={user_outfit_index.shape[1]:,}  ← bao gồm val+test')
print(f'  user_outfit_train_index  : nnz={user_outfit_train_index.shape[1]:,}  ← chỉ train (DÙNG CHO TRAINING)')
print(f'  ✅ Train-only graph built. Val+test edges đã bị loại khỏi graph propagation.')


=== STAGE 4C: Tạo split files ===
  🗑️  Đã xóa file cũ: train_rec.txt
  🗑️  Đã xóa file cũ: val_rec.txt
  🗑️  Đã xóa file cũ: test_rec.txt
  🗑️  Đã xóa file cũ: train_fitb.txt
  🗑️  Đã xóa file cũ: val_fitb.txt
  🗑️  Đã xóa file cũ: test_fitb.txt
  Rec split (users)        → train: 25,263 | val: 25,263 | test: 25,263
  Rec split (interactions) → train: 76,508 (60.0%) | val: 25,514 (20.0%) | test: 25,514 (20.0%)
  (trong đó 0 users chỉ có <3 interactions → toàn bộ vào train)

  🔧 Building train-only user_outfit graph (no val/test edges)...
  user_outfit_index (full) : nnz=127,536  ← bao gồm val+test
  user_outfit_train_index  : nnz=76,508  ← chỉ train (DÙNG CHO TRAINING)
  ✅ Train-only graph built. Val+test edges đã bị loại khỏi graph propagation.


In [7]:
print('=== STAGE 4C-FITB: Tạo FITB files ===')

def parse_ids(s):
    s = str(s).strip()
    sep = ';' if ';' in s else ' '
    return [int(x.strip()) for x in s.split(sep) if x.strip()]


def build_fitb_files(outfit_data, item2id, parse_ids_fn,
                     train_fitb_path, val_fitb_path, test_fitb_path,
                     val_ratio=0.1, test_ratio=0.1, n_neg_outfits=3,
                     hard_neg=True, item_category_map=None, seed=42):
    """Author/v4 FITB format: pos = full outfit; neg = full outfits with one swapped item."""
    random.seed(seed)
    all_item_ids = list(item2id.keys())

    cat_to_items = defaultdict(list)
    if hard_neg and item_category_map:
        for iid in all_item_ids:
            cat = item_category_map.get(iid)
            if cat:
                cat_to_items[cat].append(iid)

    records = []
    for _, row in outfit_data.iterrows():
        items = [x for x in parse_ids_fn(str(row['items'])) if x in item2id]
        if len(items) < 2:
            continue

        mask_pos = random.randint(0, len(items) - 1)
        pos_set = set(items)
        pos_str = ','.join(map(str, items))

        neg_parts = []
        for _ in range(n_neg_outfits):
            neg = items.copy()
            pos_item = items[mask_pos]
            neg_items_pool = []
            if hard_neg and item_category_map:
                pos_cat = item_category_map.get(pos_item)
                neg_items_pool = [x for x in cat_to_items.get(pos_cat, []) if x not in pos_set]
            if len(neg_items_pool) < 1:
                neg_items_pool = [x for x in all_item_ids if x not in pos_set]
            if not neg_items_pool:
                break
            neg[mask_pos] = random.choice(neg_items_pool)
            neg_parts.append(','.join(map(str, neg)))

        if len(neg_parts) < 1:
            continue

        records.append({
            'outfit_id': int(row['outfit_id']),
            'n_items': len(items),
            'mask_pos': mask_pos,
            'pos_str': pos_str,
            'neg_parts': neg_parts,
        })

    random.shuffle(records)
    n = len(records)
    n_test = max(1, round(n * test_ratio))
    n_val = max(1, round(n * val_ratio))
    train_recs = records[: n - n_val - n_test]
    val_recs = records[n - n_val - n_test: n - n_test]
    test_recs = records[n - n_test:]

    def write_fitb(path, recs):
        with open(path, 'w') as f:
            for r in recs:
                f.write(
                    f"{r['outfit_id']};{r['n_items']};{r['mask_pos']};"
                    f"{r['pos_str']};{';'.join(r['neg_parts'])}\n"
                )

    write_fitb(train_fitb_path, train_recs)
    write_fitb(val_fitb_path, val_recs)
    write_fitb(test_fitb_path, test_recs)
    return len(train_recs), len(val_recs), len(test_recs)


item_category_map = {int(k): v for k, v in zip(item_data['item_id'], item_data['category'])}

for f in [TRAIN_FITB, VAL_FITB, TEST_FITB]:
    if os.path.exists(f):
        os.remove(f)

n_tr, n_va, n_te = build_fitb_files(
    outfit_data, item2id, parse_ids,
    TRAIN_FITB, VAL_FITB, TEST_FITB,
    n_neg_outfits=3, hard_neg=True,
    item_category_map=item_category_map,
)
print(f'  FITB split → train: {n_tr:,} | val: {n_va:,} | test: {n_te:,}')
print('  Format: full pos outfit + full neg outfits (one hard-swapped item)')
print('✅ FITB files sẵn sàng!')


=== STAGE 4C-FITB: Tạo FITB files ===
  FITB split → train: 5,298 | val: 662 | test: 662
  Format: full pos outfit + full neg outfits (one hard-swapped item)
✅ FITB files sẵn sàng!


## Stage 4D — Dataset & DataLoader

In [8]:
print('=== STAGE 4D: Dataset & DataLoader ===')

class OutfitRecommendationDataset(Dataset):
    """BPR pairs with optional multi-negative sampling."""
    def __init__(self, rec_file, user2id, outfit2id, user_pos_dict=None,
                 num_outfits=None, neg_per_pos=1, seed=42):
        self.samples = []
        with open(rec_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 2:
                    continue
                uid = int(parts[0])
                for oid in map(int, parts[1:]):
                    u_idx = user2id.get(uid, -1)
                    o_idx = outfit2id.get(oid, -1)
                    if u_idx != -1 and o_idx != -1:
                        self.samples.append((u_idx, o_idx))
        self.user_pos_dict = user_pos_dict or {}
        self.num_outfits = num_outfits or 0
        self.neg_per_pos = neg_per_pos
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        u, o = self.samples[idx]
        out = {
            'user_idx': torch.tensor(u, dtype=torch.long),
            'outfit_idx': torch.tensor(o, dtype=torch.long),
        }
        if self.neg_per_pos > 0 and self.num_outfits > 0:
            pos_set = self.user_pos_dict.get(u, set())
            negs = []
            for _ in range(self.neg_per_pos):
                n = int(self.rng.integers(0, self.num_outfits))
                while n in pos_set:
                    n = int(self.rng.integers(0, self.num_outfits))
                negs.append(n)
            out['neg_outfit_idx'] = torch.tensor(negs, dtype=torch.long)
        return out


class CompatibilityDataset(Dataset):
    """FITB: parts[3]=pos outfit, parts[4:]=neg outfits (full item lists)."""
    def __init__(self, fitb_file, item2id, max_items=MAX_OUTFIT_ITEMS_FOR_COMP, seed=42):
        self.samples = []
        self.max_items = max_items
        self.rng = np.random.default_rng(seed)
        with open(fitb_file, 'r') as f:
            for line in f:
                parts = line.strip().split(';')
                if len(parts) < 5:
                    continue
                pos_ids = [item2id.get(int(i), -1) for i in parts[3].split(',') if i.strip()]
                if all(i == -1 for i in pos_ids):
                    continue
                for neg_field in parts[4:]:
                    neg_ids = [item2id.get(int(i), -1) for i in neg_field.split(',') if i.strip()]
                    if all(i == -1 for i in neg_ids):
                        continue
                    self.samples.append({'pos': pos_ids, 'neg': neg_ids})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pos = (self.samples[idx]['pos'] + [-1] * self.max_items)[:self.max_items]
        neg = (self.samples[idx]['neg'] + [-1] * self.max_items)[:self.max_items]
        return {
            'pos_item_indices': torch.tensor(pos, dtype=torch.long),
            'neg_item_indices': torch.tensor(neg, dtype=torch.long),
        }


user_pos_train_idx = defaultdict(set)
for uid, oids in user_train_outfits.items():
    u_idx = user2id.get(uid, -1)
    if u_idx == -1:
        continue
    for o in oids:
        if o in outfit2id:
            user_pos_train_idx[u_idx].add(outfit2id[o])

user_pos_all_idx = defaultdict(set)
for _, row in train_uo.iterrows():
    u_idx = user2id.get(int(row['user_id']), -1)
    o_idx = outfit2id.get(int(row['outfit_id']), -1)
    if u_idx != -1 and o_idx != -1:
        user_pos_all_idx[u_idx].add(o_idx)

train_dataset = OutfitRecommendationDataset(
    TRAIN_REC_FILE, user2id, outfit2id,
    user_pos_dict=user_pos_train_idx, num_outfits=N_OUTFITS,
    neg_per_pos=NEG_PER_POS, seed=SEED)
val_dataset = OutfitRecommendationDataset(
    VAL_REC_FILE, user2id, outfit2id,
    user_pos_dict=user_pos_all_idx, num_outfits=N_OUTFITS,
    neg_per_pos=1, seed=SEED + 1)
test_dataset = OutfitRecommendationDataset(
    TEST_REC_FILE, user2id, outfit2id,
    user_pos_dict=user_pos_all_idx, num_outfits=N_OUTFITS,
    neg_per_pos=1, seed=SEED + 2)

compat_dataset      = CompatibilityDataset(TRAIN_FITB, item2id)
compat_val_dataset  = CompatibilityDataset(VAL_FITB, item2id)
compat_test_dataset = CompatibilityDataset(TEST_FITB, item2id)

train_loader       = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=PIN_MEMORY)
val_loader         = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader        = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
compat_loader      = DataLoader(compat_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=PIN_MEMORY)
compat_val_loader  = DataLoader(compat_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
compat_test_loader = DataLoader(compat_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'  train: {len(train_dataset):,} | val: {len(val_dataset):,} | test: {len(test_dataset):,}')
print(f'  NEG_PER_POS (train)={NEG_PER_POS}  BATCH_SIZE={BATCH_SIZE}')
print('✅ Stage 4D hoàn thành!')


=== STAGE 4D: Dataset & DataLoader ===
  train: 76,508 | val: 25,514 | test: 25,514
  NEG_PER_POS (train)=3  BATCH_SIZE=512
✅ Stage 4D hoàn thành!


## Stage 4E — Định nghĩa Model H_HFGAT

In [9]:
print('=== STAGE 4E: Định nghĩa model H_HFGAT ===')

# ── MultiHeadSelfAttentionLayer ────────────────────────────────────────
class MultiHeadSelfAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads, dropout=0.2):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = out_dim // num_heads
        self.out_dim   = out_dim
        self.W         = nn.Linear(in_dim, out_dim, bias=False)
        self.bn        = nn.BatchNorm1d(out_dim)
        self.attn      = nn.Parameter(torch.Tensor(num_heads, 2 * self.head_dim))
        self.leaky     = nn.LeakyReLU(0.2)
        self.dropout   = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attn.unsqueeze(0))

    def forward(self, h, edge_index, edge_weight=None):
        N = h.size(0)
        h_proj = self.W(h).view(N, self.num_heads, self.head_dim)  # [N, H, D]
        src, dst = edge_index
        h_cat  = torch.cat([h_proj[src], h_proj[dst]], dim=-1)     # [E, H, 2D]
        e      = self.leaky((h_cat * self.attn).sum(dim=-1))        # [E, H]

        # Per-destination softmax
        alpha = torch.zeros(e.size(0), self.num_heads, device=h.device)
        for head in range(self.num_heads):
            e_h = e[:, head]
            # numerically stable softmax per dst node
            e_max = torch.zeros(N, device=h.device)
            e_max.scatter_reduce_(0, dst, e_h, reduce='amax', include_self=True)
            e_exp = torch.exp(e_h - e_max[dst])
            e_sum = torch.zeros(N, device=h.device)
            e_sum.scatter_add_(0, dst, e_exp)
            alpha[:, head] = e_exp / (e_sum[dst] + 1e-9)

        if edge_weight is not None:
            alpha = alpha * edge_weight.unsqueeze(-1)
        alpha = self.dropout(alpha)

        msg    = h_proj[src] * alpha.unsqueeze(-1)         # [E, H, D]
        h_agg  = scatter_add(msg.view(-1, self.out_dim),
                             dst, dim=0, dim_size=N)       # [N, out_dim]
        h_prime = self.bn(h_agg)
        return F.relu(h_prime)


# ── UserAttentionAggregator ────────────────────────────────────────────
class UserAttentionAggregator(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.W_proj     = nn.Linear(dim, dim, bias=False)
        self.att_vector = nn.Parameter(torch.Tensor(2 * dim))
        self.W_msg      = nn.Linear(dim, dim, bias=False)
        self.leaky      = nn.LeakyReLU(0.2)
        nn.init.xavier_uniform_(self.att_vector.unsqueeze(0))

    def forward(self, user_embs, outfit_embs_updated, user_outfit_index):
        user_idx, outfit_idx = user_outfit_index
        h_o_proj = self.W_proj(outfit_embs_updated[outfit_idx])  # [E, dim]
        h_u_proj = self.W_proj(user_embs[user_idx])              # [E, dim]
        concat   = torch.cat([h_o_proj, h_u_proj], dim=-1)       # [E, 2*dim]
        e_ou     = self.leaky((concat * self.att_vector).sum(dim=-1))  # [E]

        # ── Vectorized softmax per user (không dùng for loop) ──────────
        # Trừ max để numerical stability
        e_max = torch.zeros(user_embs.size(0), device=user_embs.device)
        e_max.scatter_reduce_(0, user_idx, e_ou, reduce='amax', include_self=True)
        e_exp = torch.exp(e_ou - e_max[user_idx])
        e_sum = torch.zeros(user_embs.size(0), device=user_embs.device)
        e_sum.scatter_add_(0, user_idx, e_exp)
        alpha = e_exp / (e_sum[user_idx] + 1e-9)  # [E]

        messages   = self.W_msg(outfit_embs_updated[outfit_idx]) * alpha.unsqueeze(-1)
        user_final = scatter_add(messages, user_idx, dim=0,
                                 dim_size=user_embs.size(0))
        return user_embs + user_final


# ── CompatibilityScorer ───────────────────────────────────────────────
class CompatibilityScorer(nn.Module):
    def __init__(self, dim, num_views=6, hidden_dim=256, dropout=0.4):
        super().__init__()
        self.W5 = nn.Linear(dim, hidden_dim)
        self.W7 = nn.Linear(dim, hidden_dim)
        self.W4 = nn.Linear(hidden_dim, num_views)
        self.W6 = nn.Linear(hidden_dim, num_views)
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, item_embs_batch, pad_mask=None):
        # item_embs_batch: [B, max_items, dim]; pad_mask: [B, max_items] True=padding
        h5 = self.dropout(F.relu(self.W5(item_embs_batch)))
        h7 = self.dropout(F.relu(self.W7(item_embs_batch)))
        attn_logits = self.W4(h5).transpose(1, 2)
        if pad_mask is not None:
            attn_logits = attn_logits.masked_fill(pad_mask.unsqueeze(1), float('-inf'))
        A = self.softmax(attn_logits)
        C = torch.tanh(self.W6(h7).transpose(1, 2))  # bounded views (author / stable BPR)
        if pad_mask is not None:
            C = C.masked_fill(pad_mask.unsqueeze(1), 0.0)
        scores = (A * C).sum(dim=-1).sum(dim=-1)
        return scores


# ── H_HFGAT ───────────────────────────────────────────────────────────
class H_HFGAT(nn.Module):
    def __init__(self, dim=64, heads=4, dropout=0.3):
        super().__init__()
        self.item_attention    = MultiHeadSelfAttentionLayer(dim, dim, heads, dropout)
        self.item_to_outfit    = nn.Linear(dim, dim)
        self.user_agg          = UserAttentionAggregator(dim)
        self.compatibility_scorer = CompatibilityScorer(dim=dim, dropout=min(0.5, dropout + 0.1))
        self.dropout           = nn.Dropout(dropout)
        self.activation        = nn.LeakyReLU(0.2)

    def forward(self,
                item_embs,   item_item_edge_index,   item_item_weight,
                outfit_embs, outfit_item_edge_index, outfit_item_weight,
                user_embs,   user_outfit_edge_index, user_outfit_weight):
        # ── Item-level GAT ──
        item_upd = self.item_attention(item_embs, item_item_edge_index, item_item_weight)
        item_upd = F.normalize(self.dropout(item_upd), p=2, dim=-1)

        # ── Outfit aggregation (sum of updated item embs) ──
        valid = ((outfit_item_edge_index[0] < outfit_embs.size(0)) &
                 (outfit_item_edge_index[1] < item_upd.size(0)))
        if not valid.all():
            outfit_item_edge_index = outfit_item_edge_index[:, valid]
            outfit_item_weight = outfit_item_weight[valid] if outfit_item_weight is not None else None

        outfit_agg = scatter_add(
            src=item_upd[outfit_item_edge_index[1]],
            index=outfit_item_edge_index[0],
            dim=0, dim_size=outfit_embs.size(0)
        )
        outfit_upd = self.dropout(self.item_to_outfit(outfit_agg))

        # ── User aggregation ──
        user_upd = self.user_agg(user_embs, outfit_upd, user_outfit_edge_index)

        return item_upd, outfit_upd, user_upd

    def score_recommendation(self, user_emb, outfit_emb):
        u = F.normalize(user_emb, p=2, dim=-1)
        o = F.normalize(outfit_emb, p=2, dim=-1)
        return torch.sum(u * o, dim=-1)

    def score_compatibility_outfit(self, item_embs, item_indices):
        """Score outfit compatibility from base item embeddings (not GAT item_upd)."""
        pad_mask = item_indices < 0
        safe_idx = item_indices.clamp_min(0)
        x = F.normalize(item_embs[safe_idx], p=2, dim=-1)
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        return self.compatibility_scorer(x, pad_mask=pad_mask)


print('✅ Stage 4E hoàn thành! Model classes defined.')

=== STAGE 4E: Định nghĩa model H_HFGAT ===
✅ Stage 4E hoàn thành! Model classes defined.


## Stage 4F — Loss functions & Evaluation

In [10]:
print('=== STAGE 4F: Loss & Evaluation ===')

def bpr_loss(pos_score, neg_score, margin=0.0):
    return -torch.mean(F.logsigmoid(pos_score - neg_score - margin))


def compat_bpr_loss(pos_score, neg_score):
    return bpr_loss(pos_score, neg_score, margin=COMPAT_BPR_MARGIN)


def compat_item_embs(item_embs_t):
    """Compat path: base item embs only; optional detach so only scorer learns."""
    return item_embs_t.detach() if COMPAT_DETACH_INPUT else item_embs_t


def bpr_loss_multi(pos_score, neg_scores, margin=0.0):
    if neg_scores.dim() == 1:
        return bpr_loss(pos_score, neg_scores, margin=margin)
    losses = [bpr_loss(pos_score, neg_scores[:, k], margin=margin) for k in range(neg_scores.size(1))]
    return sum(losses) / len(losses)


@torch.no_grad()
def eval_compat_metrics(model, item_embs, loader):
    total_loss, total_margin, n_samples, n_correct = 0.0, 0.0, 0, 0
    for batch in loader:
        pos_idx = batch['pos_item_indices'].to(item_embs.device)
        neg_idx = batch['neg_item_indices'].to(item_embs.device)
        pos_sc = model.score_compatibility_outfit(item_embs, pos_idx)
        neg_sc = model.score_compatibility_outfit(item_embs, neg_idx)
        bs = pos_sc.size(0)
        total_loss += float(compat_bpr_loss(pos_sc, neg_sc).item()) * bs
        total_margin += float((pos_sc - neg_sc).sum().item())
        n_correct += int((pos_sc > neg_sc).sum().item())
        n_samples += bs
    if n_samples == 0:
        return float('nan'), float('nan'), float('nan')
    return total_loss / n_samples, n_correct / n_samples, total_margin / n_samples


@torch.no_grad()
def eval_rec_bpr_loss(model, loader, item_upd, outfit_upd, user_upd):
    total_loss, n_samples = 0.0, 0
    for batch in loader:
        u_idx = batch['user_idx'].to(item_upd.device)
        o_idx = batch['outfit_idx'].to(item_upd.device)
        pos_sc = model.score_recommendation(user_upd[u_idx], outfit_upd[o_idx])
        if 'neg_outfit_idx' in batch:
            neg_idx = batch['neg_outfit_idx'].to(item_upd.device)
            neg_sc = model.score_recommendation(user_upd[u_idx], outfit_upd[neg_idx])
        else:
            neg_o = torch.randint(outfit_upd.size(0), o_idx.shape, device=item_upd.device)
            neg_sc = model.score_recommendation(user_upd[u_idx], outfit_upd[neg_o])
        bs = u_idx.size(0)
        total_loss += float(bpr_loss(pos_sc, neg_sc).item()) * bs
        n_samples += bs
    return total_loss / max(n_samples, 1)


@torch.no_grad()
def evaluate_recommendation(model, loader,
                             item_embs, outfit_embs, user_embs,
                             item_item_index, item_item_weight,
                             outfit_item_index, outfit_item_weight,
                             user_outfit_index, user_outfit_weight,
                             device, k=TOP_K,
                             user_known_outfits=None):
    model.eval()
    item_upd, outfit_upd, user_upd = model(
        item_embs, item_item_index, item_item_weight,
        outfit_embs, outfit_item_index, outfit_item_weight,
        user_embs, user_outfit_index, user_outfit_weight,
    )
    metrics = {'HR@K': [], 'NDCG@K': [], 'MRR@K': [], 'Precision@K': [], 'Recall@K': [], 'AUC': []}
    eval_users = 0
    true_set_sizes = []

    for batch in loader:
        user_idx = batch['user_idx'].to(device)
        outfit_idx = batch['outfit_idx'].to(device)
        for u in torch.unique(user_idx):
            eval_users += 1
            u_int = u.item()
            mask = user_idx == u
            pos_oids = outfit_idx[mask]
            n_pos = len(pos_oids)
            true_set_sizes.append(n_pos)
            pos_oids_set = set(pos_oids.cpu().tolist())
            pos_sc = model.score_recommendation(
                user_upd[u].repeat(n_pos, 1), outfit_upd[pos_oids])
            labels = torch.ones(n_pos, device=device)

            excluded = pos_oids_set.copy()
            if user_known_outfits is not None:
                excluded |= user_known_outfits.get(u_int, set())

            neg_oids_list = []
            tries = 0
            while len(neg_oids_list) < EVAL_NEG_SAMPLES and tries < EVAL_NEG_SAMPLES * 10:
                for c in torch.randint(outfit_upd.size(0), (EVAL_NEG_SAMPLES,), device=device).tolist():
                    if c not in excluded:
                        neg_oids_list.append(c)
                    if len(neg_oids_list) >= EVAL_NEG_SAMPLES:
                        break
                tries += 1
            while len(neg_oids_list) < EVAL_NEG_SAMPLES:
                neg_oids_list.append(random.randint(0, outfit_upd.size(0) - 1))

            neg_oids = torch.tensor(neg_oids_list[:EVAL_NEG_SAMPLES], dtype=torch.long, device=device)
            neg_sc = model.score_recommendation(
                user_upd[u].repeat(EVAL_NEG_SAMPLES, 1), outfit_upd[neg_oids])

            scores_all = torch.cat([pos_sc, neg_sc])
            labels_all = torch.cat([labels, torch.zeros(EVAL_NEG_SAMPLES, device=device)])

            sorted_idx = torch.argsort(scores_all, descending=True)
            first_hit_rank = None
            for r, idx in enumerate(sorted_idx.tolist(), start=1):
                if labels_all[idx].item() > 0:
                    first_hit_rank = r
                    break
            mrr = 1.0 / first_hit_rank if first_hit_rank else 0.0

            _, topk_idx = torch.topk(scores_all, k)
            hits = labels_all[topk_idx]
            hr = (hits.sum() > 0).float().item()
            prec = hits.sum().item() / k
            rec = hits.sum().item() / max(labels.sum().item(), 1)
            dcg = (hits / torch.log2(torch.arange(2, 2 + k, device=device, dtype=torch.float32))).sum().item()
            nh = min(int(labels.sum().item()), k)
            idcg = (torch.ones(nh, device=device) / torch.log2(
                torch.arange(2, 2 + nh, device=device, dtype=torch.float32))).sum().item() if nh > 0 else 0.0
            ndcg = dcg / idcg if idcg > 0 else 0.0
            try:
                auc = roc_auc_score(labels_all.cpu().numpy(), scores_all.cpu().numpy())
            except Exception:
                auc = float('nan')

            metrics['HR@K'].append(hr)
            metrics['NDCG@K'].append(ndcg)
            metrics['MRR@K'].append(mrr)
            metrics['Precision@K'].append(prec)
            metrics['Recall@K'].append(rec)
            metrics['AUC'].append(auc)

    out = {m: float(np.nanmean(v)) for m, v in metrics.items()}
    out['eval_users'] = eval_users
    out['avg_val_outfits_per_user'] = float(np.mean(true_set_sizes)) if true_set_sizes else 0.0
    out[f'precision_ceiling_perfect@{k}'] = float(
        np.mean([min(k, ts) / k for ts in true_set_sizes])
    ) if true_set_sizes else 0.0
    return out


def print_epoch_summary(row, epochs_total, patience_limit):
    epoch = row['epoch']
    lr = row.get('_lr', 0.0)
    pat = row.get('_patience', 0)
    print(f'\nEpoch {epoch}/{epochs_total}  [LR={lr:.2e}  patience={pat}/{patience_limit}]', flush=True)
    print(f'  train_loss      : {row["train_loss"]:.4f}', flush=True)
    print(f'  train_rec_loss  : {row["train_rec_loss"]:.4f}', flush=True)
    print(f'  train_comp_loss : {row["train_comp_loss"]:.4f}', flush=True)
    print(f'  val_rec_loss    : {row["val_rec_loss"]:.4f}', flush=True)
    print(f'  val_comp_loss   : {row["val_comp_loss"]:.4f}  compat_acc={row["compat_acc"]:.4f}  margin={row.get("compat_margin", float("nan")):.4f}', flush=True)
    print(f'  val_total_loss  : {row["val_total_loss"]:.4f}  (= rec + {LAMBDA_COMP}*comp)', flush=True)
    print(f'  eval_users      : {row["eval_users"]}', flush=True)
    print(f'  AUC             : {row["AUC"]:.4f}', flush=True)
    print(f'  HR@10           : {row["HR@10"]:.4f}', flush=True)
    print(f'  Recall@10       : {row["Recall@10"]:.4f}', flush=True)
    print(f'  NDCG@10         : {row["NDCG@10"]:.4f}', flush=True)
    print(f'  MRR@10          : {row["MRR@10"]:.4f}', flush=True)
    print(f'  Precision@10    : {row["Precision@10"]:.4f}', flush=True)
    print(f'  avg_val_outfits : {row["avg_val_outfits"]:.2f}', flush=True)
    print(f'  prec_ceiling@10 : {row["prec_ceiling@10"]:.4f}  (max P@10 if perfect rank)', flush=True)
    print('-' * 60, flush=True)


user_known_outfits_idx = {}
for uid, oids in user_train_outfits.items():
    u_idx = user2id.get(uid, -1)
    if u_idx == -1:
        continue
    user_known_outfits_idx[u_idx] = {outfit2id[o] for o in oids if o in outfit2id}

print(f'  user_known_outfits_idx: {len(user_known_outfits_idx):,}')
print(f'  compat: base_item_embs only, detach={COMPAT_DETACH_INPUT}, margin={COMPAT_BPR_MARGIN}')
print('✅ Stage 4F hoàn thành!')


=== STAGE 4F: Loss & Evaluation ===
  user_known_outfits_idx: 25,263
  compat: base_item_embs only, detach=True, margin=0.0
✅ Stage 4F hoàn thành!


## Stage 4G — Khởi tạo model & optimizer

In [11]:
print('=== STAGE 4G: Khởi tạo model ===')

SAVE_PATH = str(BEST_STATE_PATH)

model = H_HFGAT(dim=EMBED_DIM, heads=NUM_HEADS, dropout=DROPOUT).to(device)

if LEARNABLE_EMBEDDINGS:
    item_embs_t = nn.Parameter(item_embs.clone())
    outfit_embs_t = nn.Parameter(outfit_embs.clone())
    user_embs_t = nn.Parameter(user_embs.clone())
    print('  Embeddings: learnable (fine-tune during training)')
else:
    item_embs_t = item_embs.clone().detach()
    outfit_embs_t = outfit_embs.clone().detach()
    user_embs_t = user_embs.clone().detach()
    print('  Embeddings: frozen')

compat_param_ids = {id(p) for p in model.compatibility_scorer.parameters()}
core_params = [p for n, p in model.named_parameters() if id(p) not in compat_param_ids]
compat_params = list(model.compatibility_scorer.parameters())
embed_params = [item_embs_t, outfit_embs_t, user_embs_t] if LEARNABLE_EMBEDDINGS else []

OPTIM_PARAMS = core_params + compat_params + embed_params
optimizer = torch.optim.AdamW([
    {'params': core_params + embed_params, 'lr': LR},
    {'params': compat_params, 'lr': LR * COMPAT_LR_MULT},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=SCHEDULER_PATIENCE, min_lr=1e-6,
)

n_params = sum(p.numel() for p in model.parameters())
print(f'  Model params: {n_params:,}')
print(f'  Optimizer: core+emb LR={LR}, compat_scorer LR={LR * COMPAT_LR_MULT}')
print('✅ Stage 4G hoàn thành!')


=== STAGE 4G: Khởi tạo model ===
  Embeddings: learnable (fine-tune during training)
  Model params: 53,196
  Optimizer: core+emb LR=0.001, compat_scorer LR=0.001
✅ Stage 4G hoàn thành!


## Stage 4H — Training loop

In [12]:
print('=== STAGE 4H: Training ===')
print(f'🚀 Starting training loop with {EPOCHS} epochs (early-stop patience={PATIENCE})...')

history = defaultdict(list)
history_rows = []
best_val_hr = 0.0
best_epoch = -1
best_metrics = {}
best_state_dict = None
early_stop_count = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = total_rec = total_comp = 0.0
    n_batches = 0
    compat_iter = iter(compat_loader)

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for batch in pbar:
        optimizer.zero_grad()
        item_upd, outfit_upd, user_upd = model(
            item_embs_t, item_item_index, item_item_weight,
            outfit_embs_t, outfit_item_index, outfit_item_weight,
            user_embs_t, user_outfit_train_index, user_outfit_train_weight,
        )
        u_idx = batch['user_idx'].to(device)
        o_idx = batch['outfit_idx'].to(device)
        pos_sc = model.score_recommendation(user_upd[u_idx], outfit_upd[o_idx])

        if 'neg_outfit_idx' in batch:
            neg_idx = batch['neg_outfit_idx'].to(device)
            if neg_idx.dim() == 2:
                neg_scores = torch.stack([
                    model.score_recommendation(user_upd[u_idx], outfit_upd[neg_idx[:, k]])
                    for k in range(neg_idx.size(1))
                ], dim=1)
                loss_rec = bpr_loss_multi(pos_sc, neg_scores)
            else:
                loss_rec = bpr_loss(pos_sc, model.score_recommendation(user_upd[u_idx], outfit_upd[neg_idx]))
        else:
            neg_o = torch.randint(outfit_upd.size(0), o_idx.shape, device=device)
            loss_rec = bpr_loss(pos_sc, model.score_recommendation(user_upd[u_idx], outfit_upd[neg_o]))

        try:
            cb = next(compat_iter)
        except StopIteration:
            compat_iter = iter(compat_loader)
            cb = next(compat_iter)

        pi = cb['pos_item_indices'].to(device)
        ni = cb['neg_item_indices'].to(device)
        compat_embs = compat_item_embs(item_embs_t)
        loss_comp = compat_bpr_loss(
            model.score_compatibility_outfit(compat_embs, pi),
            model.score_compatibility_outfit(compat_embs, ni),
        )
        loss = loss_rec + LAMBDA_COMP * loss_comp
        loss.backward()
        torch.nn.utils.clip_grad_norm_(OPTIM_PARAMS, max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_rec += loss_rec.item()
        total_comp += loss_comp.item()
        n_batches += 1
        pbar.set_postfix({
            'loss': f'{total_loss / n_batches:.4f}',
            'rec': f'{total_rec / n_batches:.4f}',
            'comp': f'{total_comp / n_batches:.4f}',
        })

    train_loss = total_loss / max(n_batches, 1)
    train_rec_loss = total_rec / max(n_batches, 1)
    train_comp_loss = total_comp / max(n_batches, 1)

    model.eval()
    with torch.no_grad():
        item_upd_v, outfit_upd_v, user_upd_v = model(
            item_embs_t, item_item_index, item_item_weight,
            outfit_embs_t, outfit_item_index, outfit_item_weight,
            user_embs_t, user_outfit_train_index, user_outfit_train_weight,
        )
        val_rec_loss = eval_rec_bpr_loss(model, val_loader, item_upd_v, outfit_upd_v, user_upd_v)
        val_comp_loss, val_compat_acc, compat_margin = eval_compat_metrics(
            model, item_embs_t, compat_val_loader)

    do_eval = (epoch % EVAL_EVERY == 0)
    if do_eval:
        val_rec_metrics = evaluate_recommendation(
            model, val_loader, item_embs_t, outfit_embs_t, user_embs_t,
            item_item_index, item_item_weight, outfit_item_index, outfit_item_weight,
            user_outfit_train_index, user_outfit_train_weight, device,
            user_known_outfits=user_known_outfits_idx,
        )
    else:
        val_rec_metrics = {k: float('nan') for k in [
            'HR@K', 'NDCG@K', 'MRR@K', 'Precision@K', 'Recall@K', 'AUC',
            'eval_users', 'avg_val_outfits_per_user', f'precision_ceiling_perfect@{TOP_K}',
        ]}
        compat_margin = float('nan')

    val_total_loss = val_rec_loss + LAMBDA_COMP * val_comp_loss
    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'train_rec_loss': train_rec_loss,
        'train_comp_loss': train_comp_loss,
        'val_total_loss': val_total_loss,
        'val_rec_loss': val_rec_loss,
        'val_comp_loss': val_comp_loss,
        'val_bpr_loss': val_rec_loss,
        'compat_acc': val_compat_acc,
        'eval_users': int(val_rec_metrics.get('eval_users', 0) or 0),
        'AUC': float(val_rec_metrics.get('AUC', float('nan'))),
        'HR@10': float(val_rec_metrics.get('HR@K', float('nan'))),
        'Recall@10': float(val_rec_metrics.get('Recall@K', float('nan'))),
        'NDCG@10': float(val_rec_metrics.get('NDCG@K', float('nan'))),
        'MRR@10': float(val_rec_metrics.get('MRR@K', float('nan'))),
        'Precision@10': float(val_rec_metrics.get('Precision@K', float('nan'))),
        'avg_val_outfits': float(val_rec_metrics.get('avg_val_outfits_per_user', float('nan'))),
        'prec_ceiling@10': float(val_rec_metrics.get(f'precision_ceiling_perfect@{TOP_K}', float('nan'))),
        'compat_margin': compat_margin if do_eval else float('nan'),
    }
    history_rows.append(dict(row))

    history['train_loss'].append(train_loss)
    history['train_rec'].append(train_rec_loss)
    history['train_comp'].append(train_comp_loss)
    history['val_loss'].append(val_total_loss)
    history['val_rec_loss'].append(val_rec_loss)
    history['val_comp_loss'].append(val_comp_loss)
    history['val_compat_acc'].append(val_compat_acc)
    for k_, v_ in val_rec_metrics.items():
        history[f'val_{k_}'].append(v_)

    metric_key = EARLY_STOP_METRIC
    current_metric = val_rec_metrics.get(metric_key, 0.0) if do_eval else best_val_hr

    if do_eval and current_metric > best_val_hr:
        best_val_hr = current_metric
        best_epoch = epoch
        best_state_dict = {
            'model': copy.deepcopy(model.state_dict()),
            'item_embs': item_embs_t.detach().cpu().clone(),
            'outfit_embs': outfit_embs_t.detach().cpu().clone(),
            'user_embs': user_embs_t.detach().cpu().clone(),
        }
        best_metrics = dict(row)
        torch.save(best_state_dict, SAVE_PATH)
        early_stop_count = 0
    elif do_eval:
        early_stop_count += 1

    if do_eval:
        scheduler.step(current_metric)
    else:
        scheduler.step(best_val_hr)

    row['_lr'] = optimizer.param_groups[0]['lr']
    row['_patience'] = early_stop_count
    print_epoch_summary(row, EPOCHS, PATIENCE)

    if do_eval and early_stop_count >= PATIENCE:
        print('🛑 Early stopping!', flush=True)
        break

print(f'\n✅ Training done! Best Val {EARLY_STOP_METRIC}={best_val_hr:.4f} @ epoch {best_epoch}')


=== STAGE 4H: Training ===
🚀 Starting training loop with 50 epochs (early-stop patience=10)...


Epoch 1/50: 100%|█| 150/150 [00:09<00:00, 16.31it/s, loss=0.7859, rec=



Epoch 1/50  [LR=1.00e-03  patience=0/10]
  train_loss      : 0.7859
  train_rec_loss  : 0.5804
  train_comp_loss : 0.6848
  val_rec_loss    : 0.5386
  val_comp_loss   : 0.6665  compat_acc=0.5826  margin=0.0835
  val_total_loss  : 0.7385  (= rec + 0.3*comp)
  eval_users      : 25263
  AUC             : 0.7348
  HR@10           : 0.5515
  Recall@10       : 0.5493
  NDCG@10         : 0.3017
  MRR@10          : 0.2485
  Precision@10    : 0.0555
  avg_val_outfits : 1.01
  prec_ceiling@10 : 0.1010  (max P@10 if perfect rank)
------------------------------------------------------------


Epoch 2/50: 100%|█| 150/150 [00:05<00:00, 28.42it/s, loss=0.6730, rec=



Epoch 2/50  [LR=1.00e-03  patience=0/10]
  train_loss      : 0.6730
  train_rec_loss  : 0.4747
  train_comp_loss : 0.6610
  val_rec_loss    : 0.4862
  val_comp_loss   : 0.6375  compat_acc=0.5901  margin=0.2493
  val_total_loss  : 0.6774  (= rec + 0.3*comp)
  eval_users      : 25263
  AUC             : 0.7729
  HR@10           : 0.6508
  Recall@10       : 0.6489
  NDCG@10         : 0.3659
  MRR@10          : 0.2960
  Precision@10    : 0.0655
  avg_val_outfits : 1.01
  prec_ceiling@10 : 0.1010  (max P@10 if perfect rank)
------------------------------------------------------------


...


Epoch 48/50  [LR=5.00e-04  patience=8/10]
  train_loss      : 0.4611
  train_rec_loss  : 0.3416
  train_comp_loss : 0.3983
  val_rec_loss    : 0.4005
  val_comp_loss   : 0.5224  compat_acc=0.7422  margin=0.9643
  val_total_loss  : 0.5572  (= rec + 0.3*comp)
  eval_users      : 25263
  AUC             : 0.8417
  HR@10           : 0.7684
  Recall@10       : 0.7669
  NDCG@10         : 0.4567
  MRR@10          : 0.3713
  Precision@10    : 0.0774
  avg_val_outfits : 1.01
  prec_ceiling@10 : 0.1010  (max P@10 if perfect rank)
------------------------------------------------------------


Epoch 49/50: 100%|█| 150/150 [00:05<00:00, 28.66it/s, loss=0.4608, rec



Epoch 49/50  [LR=5.00e-04  patience=9/10]
  train_loss      : 0.4608
  train_rec_loss  : 0.3423
  train_comp_loss : 0.3947
  val_rec_loss    : 0.3936
  val_comp_loss   : 0.5228  compat_acc=0.7346  margin=0.9634
  val_total_loss  : 0.5504  (= rec + 0.3*comp)
  eval_users      : 25263
  AUC             : 0.8450
  HR@10           : 0.7722
  Recall@10       : 0.7706
  NDCG@10         : 0.4675
  MRR@10          : 0.3840
  Precision@10    : 0.0778
  avg_val_outfits : 1.01
  prec_ceiling@10 : 0.1010  (max P@10 if perfect rank)
------------------------------------------------------------


Epoch 50/50: 100%|█| 150/150 [00:05<00:00, 28.66it/s, loss=0.4592, rec



Epoch 50/50  [LR=5.00e-04  patience=10/10]
  train_loss      : 0.4592
  train_rec_loss  : 0.3410
  train_comp_loss : 0.3942
  val_rec_loss    : 0.3977
  val_comp_loss   : 0.5209  compat_acc=0.7367  margin=0.9654
  val_total_loss  : 0.5540  (= rec + 0.3*comp)
  eval_users      : 25263
  AUC             : 0.8429
  HR@10           : 0.7693
  Recall@10       : 0.7677
  NDCG@10         : 0.4627
  MRR@10          : 0.3788
  Precision@10    : 0.0775
  avg_val_outfits : 1.01
  prec_ceiling@10 : 0.1010  (max P@10 if perfect rank)
------------------------------------------------------------
🛑 Early stopping!

✅ Training done! Best Val HR@K=0.7745 @ epoch 40


## Stage 4I — Evaluate trên Test set

In [13]:
print('=== STAGE 4I: Test evaluation ===')

ckpt = best_state_dict
if ckpt is None:
    ckpt = torch.load(SAVE_PATH, map_location=device)
if isinstance(ckpt, dict) and 'model' in ckpt:
    model.load_state_dict(ckpt['model'])
    if LEARNABLE_EMBEDDINGS:
        item_embs_t.data.copy_(ckpt['item_embs'].to(device))
        outfit_embs_t.data.copy_(ckpt['outfit_embs'].to(device))
        user_embs_t.data.copy_(ckpt['user_embs'].to(device))
else:
    model.load_state_dict(ckpt)
model.eval()

with torch.no_grad():
    item_upd_test, outfit_upd_test, user_upd_test = model(
        item_embs_t, item_item_index, item_item_weight,
        outfit_embs_t, outfit_item_index, outfit_item_weight,
        user_embs_t, user_outfit_train_index, user_outfit_train_weight,
    )

test_rec_metrics = evaluate_recommendation(
    model, test_loader, item_embs_t, outfit_embs_t, user_embs_t,
    item_item_index, item_item_weight, outfit_item_index, outfit_item_weight,
    user_outfit_train_index, user_outfit_train_weight, device,
    user_known_outfits=user_known_outfits_idx,
)
_, test_compat_acc, _ = eval_compat_metrics(model, item_embs_t, compat_test_loader)

print('\n📊 TEST RESULTS')
for label, key in [('HR@10', 'HR@K'), ('NDCG@10', 'NDCG@K'), ('MRR@10', 'MRR@K'),
                   ('Prec@10', 'Precision@K'), ('Rec@10', 'Recall@K'), ('AUC', 'AUC')]:
    print(f'  {label}: {test_rec_metrics[key]:.4f}')
print(f'  compat_acc: {test_compat_acc:.4f}')
print(f'  eval_users: {test_rec_metrics["eval_users"]}')

results = {
    **{f'test_{k}': v for k, v in test_rec_metrics.items()},
    'test_compat_acc': test_compat_acc,
    'best_val_hr': best_val_hr,
    'best_epoch': int(best_epoch),
    'best_val_metrics': best_metrics,
}
with open(OUTPUT_PATH + 'models/test_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'  💾 best_model.pt → {SAVE_PATH}')
print(f'  💾 test_results.json saved')
print('✅ Stage 4I hoàn thành!')


=== STAGE 4I: Test evaluation ===

📊 TEST RESULTS
  HR@10: 0.7739
  NDCG@10: 0.4662
  MRR@10: 0.3820
  Prec@10: 0.0779
  Rec@10: 0.7722
  AUC: 0.8480
  compat_acc: 0.7216
  eval_users: 25263
  💾 best_model.pt → /Users/quoctran/Documents/999999_HCMUS_Master_Exercises/HocPhan2/2_Research_Methods/hcmus-master-is-research-methods/output_fgat_active_user/models/best_model.pt
  💾 test_results.json saved
✅ Stage 4I hoàn thành!


## Stage 4J — Vẽ training curves

In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('H-HFGAT Training Curves', fontsize=14)

axes[0,0].plot(epochs_range, history['train_loss'], label='Train')
axes[0,0].plot(epochs_range, history['val_loss'],   label='Val')
axes[0,0].set_title('Total Loss'); axes[0,0].legend()

axes[0,1].plot(epochs_range, history['train_rec'], label='Train')
axes[0,1].plot(epochs_range, history['val_rec_loss'], label='Val')
axes[0,1].set_title('Rec Loss'); axes[0,1].legend()

axes[0,2].plot(epochs_range, history['train_comp'], label='Train')
axes[0,2].plot(epochs_range, history['val_comp_loss'], label='Val')
axes[0,2].set_title('Compat Loss'); axes[0,2].legend()

axes[1,0].plot(epochs_range, history['val_HR@K'], label='Val HR@10')
axes[1,0].plot(epochs_range, history['val_NDCG@K'], label='Val NDCG@10')
axes[1,0].set_title('Rec Metrics'); axes[1,0].legend()

axes[1,1].plot(epochs_range, history['val_AUC'])
axes[1,1].set_title('Val AUC')

axes[1,2].plot(epochs_range, history['val_compat_acc'])
axes[1,2].set_title('Val Compat Accuracy')

plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'models/training_curves.png', dpi=150)
plt.show()
print('  💾 Đã lưu training_curves.png')
print('\n✅ Notebook 3 hoàn thành!')

  💾 Đã lưu training_curves.png

✅ Notebook 3 hoàn thành!


/var/folders/5b/xnk1ngy13x15l67b_3vwfbgm0000gn/T/ipykernel_3253/2543673504.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
